# 08 Time Travel and RESTORE

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Query any retained version of a Delta table (<code>VERSION AS OF</code>, <code>TIMESTAMP AS OF</code>), compare versions to audit what changed, roll back with <code>RESTORE</code>, copy tables with <code>CLONE</code>, and understand the retention settings that decide how far back you can go.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Mistakes happen: a bad deploy overwrites a table, a <code>DELETE</code> has the wrong <code>WHERE</code>. With Delta you can see exactly what changed and undo it in seconds. Time travel also gives reproducible ML training sets and audit trails. The exam asks about <code>DESCRIBE HISTORY</code>, <code>VERSION AS OF</code>, <code>RESTORE</code> and how <code>VACUUM</code> limits time travel.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How time travel works</b><br>
Every commit is a version. A version is defined by the log (which files were live at that point), and the data files stay on storage until <code>VACUUM</code> deletes them. To read version N, Delta replays the log up to N and reads those files.<br><br>
| Setting (table property) | Default | Controls |<br>|---|---|---|<br>| <code>delta.logRetentionDuration</code> | 30 days | How long log entries (history) are kept |<br>| <code>delta.deletedFileRetentionDuration</code> | 7 days | How long removed data files are kept before <code>VACUUM</code> may delete them |<br><br>
You can time travel only to versions whose <b>log entries</b> and <b>data files</b> still exist. In practice that's the last 7 days by default, once <code>VACUUM</code> runs.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
At 09:10 a new version of a pipeline wrote prices in cents instead of dollars to the <code>products</code> Gold table. Dashboards showed revenue 100 times too high. The engineer ran <code>DESCRIBE HISTORY</code>, saw the bad <code>WRITE</code> at version 412, compared it with version 411 to confirm, and ran <code>RESTORE TABLE products TO VERSION AS OF 411</code>. Dashboards were correct again by 09:25, with no reprocessing.
</div>

## Syntax

```sql
DESCRIBE HISTORY t;
SELECT * FROM t VERSION AS OF 3;
SELECT * FROM t TIMESTAMP AS OF '2024-11-01 10:00:00';
SELECT * FROM t@v3;                                   -- Databricks shorthand
RESTORE TABLE t TO VERSION AS OF 3;
RESTORE TABLE t TO TIMESTAMP AS OF '2024-11-01 10:00:00';
CREATE TABLE t_copy DEEP CLONE t;                     -- independent copy (Databricks)
CREATE TABLE t_dev SHALLOW CLONE t;                   -- metadata copy that references t's files
```

```python
spark.read.option("versionAsOf", 3).table("t")
spark.read.option("timestampAsOf", "2024-11-01 10:00:00").table("t")
```

## Setup: a table with a history

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a <code>prices</code> table and changes it four times: append, update, a "bad" overwrite, and a delete.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A history with versions 0 to 4.
</div>

In [0]:
DB = "workspace.delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
import time
from pyspark.sql import functions as F

table = f"{DB}.prices"
spark.sql(f"CREATE OR REPLACE TABLE {table} (product_id INT, name STRING, price DECIMAL(10,2))")      # v0
spark.sql(f"INSERT INTO {table} VALUES (1, 'Laptop', 900.0), (2, 'Mouse', 20.0), (3, 'Desk', 250.0)")   # v1
time.sleep(1)
spark.sql(f"UPDATE {table} SET price = price * 1.1 WHERE product_id = 1")                     # v2
time.sleep(1)
spark.sql(f"INSERT OVERWRITE {table} SELECT product_id, name, price * 100 FROM {table}")      # v3: the bad deploy
time.sleep(1)
spark.sql(f"DELETE FROM {table} WHERE product_id = 3")                                        # v4

history = spark.sql(f"DESCRIBE HISTORY {table}").select("version", "timestamp", "operation", "operationParameters")
history.show(truncate=False)

## 1. Query an old version

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads version 1 with SQL and Python, and version 2 by timestamp.<br><br>
<b>Why it matters</b><br>Old versions behave like normal tables: you can filter, join and aggregate them. <code>TIMESTAMP AS OF</code> uses the latest version committed at or before the given time.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Version 1: the 3 original prices. Version 2 (by timestamp): the Laptop at 990.0. Current: 2 products with prices multiplied by 100.
</div>

In [0]:
spark.sql(f"SELECT * FROM {table} VERSION AS OF 1 ORDER BY product_id").show()

# Take the commit timestamp as a string straight from Spark (avoids time zone conversions in Python)
ts_v2 = history.filter("version = 2").selectExpr("CAST(timestamp AS STRING) AS ts").first()["ts"]
print("version 2 was committed at", ts_v2)
spark.sql(f"SELECT * FROM {table} TIMESTAMP AS OF '{ts_v2}' ORDER BY product_id").show()

spark.table(table).orderBy("product_id").show()

## 2. Audit: what changed between two versions?

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Compares version 2 with version 3 using <code>EXCEPT</code>, and counts the rows per version.<br><br>
<b>Why it matters</b><br>Before restoring, confirm what the bad change did. Set operations between versions answer "which rows changed?" precisely. (Lesson 11's change data feed does this incrementally and at scale.)
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Rows in version 2 but not in version 3 (the old prices), and rows in version 3 but not in version 2 (the prices × 100). Then the row count per version.
</div>

In [0]:
print("removed or changed by version 3:")
spark.sql(f"SELECT * FROM {table} VERSION AS OF 2 EXCEPT SELECT * FROM {table} VERSION AS OF 3").show()
print("added or changed by version 3:")
spark.sql(f"SELECT * FROM {table} VERSION AS OF 3 EXCEPT SELECT * FROM {table} VERSION AS OF 2").show()

for v in range(5):
    print(f"version {v}: {spark.read.option('versionAsOf', v).table(table).count()} rows")

## 3. Roll back with RESTORE

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Restores the table to version 2, the last good state, then checks the data and the history.<br><br>
<b>Why it matters</b><br><code>RESTORE</code> makes an old version the <b>current</b> one by writing a <b>new</b> commit. Nothing is erased: the bad versions stay in the history and can still be queried. Restoring is fast, because it only changes which files are live (where the files still exist).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The current data matches version 2 (3 products, Laptop at 990.0). The history shows a new version 5 with operation <code>RESTORE</code> and metrics such as <code>num_restored_files</code>.
</div>

In [0]:
spark.sql(f"RESTORE TABLE {table} TO VERSION AS OF 2").show(truncate=False)
spark.table(table).orderBy("product_id").show()
spark.sql(f"DESCRIBE HISTORY {table} LIMIT 2").select("version", "operation", "operationParameters").show(truncate=False)

## 4. Reproducible reads for ML and reports

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Pins a version number and reads it later, after the table has changed again.<br><br>
<b>Why it matters</b><br>Training a model or producing a regulatory report on a <b>pinned version</b> makes the result reproducible: anyone can re-read exactly the same data later, as long as the version is retained.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The pinned version still returns the same total after a new change to the table.
</div>

In [0]:
pinned = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["version"]
total_then = spark.read.option("versionAsOf", pinned).table(table).agg(F.sum("price")).first()[0]

spark.sql(f"UPDATE {table} SET price = 1.0")       # the table changes afterwards

total_later = spark.read.option("versionAsOf", pinned).table(table).agg(F.sum("price")).first()[0]
print(f"pinned version {pinned}: total then {total_then:.2f}, total read later {total_later:.2f}, same: {total_then == total_later}")

## 5. CLONE: copies for testing and backup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a shallow clone (metadata only, sharing data files) and, on Databricks, a deep clone (full independent copy).<br><br>
<b>Why it matters</b>
<ul><li><b>Shallow clone</b>: instant, cheap copy for testing changes or experiments. It references the source's files, so it breaks if the source files are vacuumed</li><li><b>Deep clone</b>: copies data and metadata. It's independent of the source, so it's used for backups, migrations and sharing with other environments</li></ul>
Both clones have their own history, starting at version 0.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Deep clone is a Databricks feature, and Unity Catalog supports shallow clones of managed tables in recent runtimes. If a clone statement isn't available in your workspace, read the output message and move on.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The clone has the same rows as the source, and its own history with a <code>CLONE</code> operation.
</div>

In [0]:
for kind in ["SHALLOW", "DEEP"]:
    clone = f"{DB}.prices_{kind.lower()}_clone"
    try:
        spark.sql(f"CREATE OR REPLACE TABLE {clone} {kind} CLONE {table}")
        print(f"{kind} clone rows:", spark.table(clone).count())
        spark.sql(f"DESCRIBE HISTORY {clone}").select("version", "operation").show()
        spark.sql(f"DROP TABLE {clone}")
    except Exception as e:
        print(f"{kind} CLONE not available here:", type(e).__name__)

## 6. Time travel limits: VACUUM

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Explains why very old versions can become unreadable, and how to check the retention settings.<br><br>
<b>Why it matters</b><br><code>VACUUM</code> permanently deletes data files that are no longer referenced by the current version and are older than <code>delta.deletedFileRetentionDuration</code> (7 days). After that, versions that needed those files fail with a file-not-found error. Lesson 09 demonstrates it.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The table's properties. If the retention properties aren't listed, the defaults apply: 30 days of log, 7 days of deleted files.
</div>

In [0]:
spark.sql(f"SHOW TBLPROPERTIES {table}").show(truncate=False)

## Under the hood

```
SELECT ... VERSION AS OF 2
  -> read the log up to 00000000000000000002.json (from a checkpoint if available)
  -> list of files live at version 2 -> read them (they still exist unless vacuumed)

RESTORE TO VERSION 2
  -> compute the file set of version 2 vs the current version
  -> new commit: add the files of v2 that aren't live, remove the files that aren't in v2
  -> history continues: v5 = RESTORE (v3 and v4 are still queryable)
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> `RESTORE` and `CLONE` are commits. Time-travel reads are normal queries.

In [0]:
spark.sql(f"DESCRIBE HISTORY {table}").select("version", "operation").show()

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {table}")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>VersionNotFound</code> / <code>DELTA_FILE_NOT_FOUND</code> when time travelling</b><br>
The version is older than the log retention, or its files were vacuumed. Increase the retention properties <b>before</b> you need them, if you require longer history.<br><br>
**⛔ Problem: <code>TIMESTAMP AS OF</code> a time before the table existed**<br>It fails. Pick a timestamp after the first commit.<br><br>
**⛔ Problem: expecting <code>RESTORE</code> to delete the bad versions**<br><code>RESTORE</code> adds a new version. The history (including bad versions) remains.<br><br>
<b>⛔ Problem: a shallow clone breaks after maintenance on the source</b><br>Shallow clones reference the source's files, and <code>VACUUM</code> on the source can delete them. Use a deep clone for anything long-lived.<br><br>
<b>⛔ Problem: using time travel as a backup strategy</b><br>History is limited by retention and lives in the same storage. Use deep clones or separate backups for disaster recovery.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is time travel in Delta Lake?</b><br>
The ability to query a table as of an earlier version number or timestamp, using <code>VERSION AS OF</code> or <code>TIMESTAMP AS OF</code>, because the log records which files made up each version and old files are retained until <code>VACUUM</code>.<br><br>

<b>🎤 2. How do you undo a bad write to a Delta table?</b><br>
Find the last good version with <code>DESCRIBE HISTORY</code>, confirm by querying it, then run <code>RESTORE TABLE t TO VERSION AS OF n</code>. That creates a new version equal to the old state, and the history is preserved.<br><br>

<b>🎤 3. What limits how far back you can time travel?</b><br>
Log retention (<code>delta.logRetentionDuration</code>, 30 days by default) and file retention (<code>delta.deletedFileRetentionDuration</code>, 7 days by default). After <code>VACUUM</code> deletes files older than the file retention, versions that need them can't be read.<br><br>

<b>🎤 4. What is the difference between a shallow clone and a deep clone?</b><br>
A shallow clone copies only the metadata and references the source's data files, so it's fast and cheap but depends on the source files. A deep clone copies both data and metadata into an independent table.<br><br>

<b>🎤 5. Give use cases for time travel.</b><br>
Rolling back bad writes, auditing what changed and when, reproducing ML training data or reports on a pinned version, and debugging pipelines by comparing versions.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer accidentally deleted rows from a Delta table 2 hours ago. What is the fastest way to recover the table?</b><br>
A. Reload all data from the source system<br>
B. <code>RESTORE TABLE t TO VERSION AS OF &lt;version before the delete&gt;</code><br>
C. <code>UNDROP TABLE t</code><br>
D. Run <code>VACUUM</code> to recover files<br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>UNDROP</code> is for dropped tables. <code>VACUUM</code> deletes files, it doesn't recover them.</details><br><br>

<b>Q2. Which command shows each version of a Delta table with the operation, user and timestamp?</b><br>
A. <code>SHOW VERSIONS t</code><br>
B. <code>DESCRIBE HISTORY t</code><br>
C. <code>DESCRIBE DETAIL t</code><br>
D. <code>SHOW TBLPROPERTIES t</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. After running <code>VACUUM</code> with the default retention, a query using <code>VERSION AS OF</code> on a 30-day-old version fails. Why?</b><br>
A. Time travel is limited to 24 hours<br>
B. The data files needed by that version were older than the 7-day file retention and were deleted<br>
C. The transaction log was converted to Parquet<br>
D. <code>VACUUM</code> resets the version numbers<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a table <code>inventory (sku STRING, qty INT)</code> with 3 rows, then make 3 changes (an update, a delete and an append)</li><li>Show how <code>qty</code> for one SKU changed across every version</li><li>Simulate a bad <code>UPDATE</code> that sets every <code>qty</code> to 0, then restore the previous version</li><li>Confirm that the bad version is still in the history and still queryable</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
inv = f"{DB}.inventory"

# 1. Table and three changes
spark.sql(f"CREATE OR REPLACE TABLE {inv} (sku STRING, qty INT)")
spark.sql(f"INSERT INTO {inv} VALUES ('A', 10), ('B', 5), ('C', 0)")
spark.sql(f"UPDATE {inv} SET qty = 7 WHERE sku = 'A'")
spark.sql(f"DELETE FROM {inv} WHERE qty = 0")
spark.sql(f"INSERT INTO {inv} VALUES ('D', 3)")

# 2. qty of 'A' across versions
latest = spark.sql(f"DESCRIBE HISTORY {inv} LIMIT 1").first()["version"]
for v in range(1, latest + 1):
    row = spark.read.option("versionAsOf", v).table(inv).filter("sku = 'A'").first()
    print(f"version {v}: qty of A = {row['qty'] if row else None}")

# 3. Bad update, then restore
spark.sql(f"UPDATE {inv} SET qty = 0")
bad = spark.sql(f"DESCRIBE HISTORY {inv} LIMIT 1").first()["version"]
spark.sql(f"RESTORE TABLE {inv} TO VERSION AS OF {bad - 1}")
spark.table(inv).orderBy("sku").show()

# 4. The bad version is still there
spark.sql(f"SELECT * FROM {inv} VERSION AS OF {bad} ORDER BY sku").show()
spark.sql(f"DESCRIBE HISTORY {inv}").select("version", "operation").show()
spark.sql(f"DROP TABLE {inv}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Every commit is a version. <code>DESCRIBE HISTORY</code> lists them with operations and metrics</li><li>Read old data with <code>VERSION AS OF</code>, <code>TIMESTAMP AS OF</code>, or the <code>versionAsOf</code> / <code>timestampAsOf</code> options</li><li>Compare versions with <code>EXCEPT</code> to audit changes</li><li><code>RESTORE</code> rolls back by adding a new version. The history is preserved</li><li>Shallow clone = metadata copy referencing the source's files. Deep clone = independent copy</li><li>Retention: 30 days of log, 7 days of removed files. <code>VACUUM</code> ends time travel beyond that</li></ul>
</div>

| Task | Code |
|---|---|
| History | `DESCRIBE HISTORY t` |
| Old version | `SELECT * FROM t VERSION AS OF 5` |
| By time | `SELECT * FROM t TIMESTAMP AS OF '2024-11-01'` |
| Python | `spark.read.option("versionAsOf", 5).table("t")` |
| Roll back | `RESTORE TABLE t TO VERSION AS OF 5` |
| Copy | `CREATE TABLE c DEEP CLONE t`, `... SHALLOW CLONE t` |

## Git commit

```
git add 03_delta_lake/08_time_travel_restore.ipynb
git commit -m "add 03_08 time travel restore notebook"
```